In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from itertools import groupby
from operator import itemgetter

In [2]:
# Configuration
plot_streamflow_series = True  # Plot individual streamflow time series

In [3]:
# ---------------- Paths ----------------

# Define paths relative to base path
path_to_xls_files = Path("./streamflow_timeseries")  

#File to match gaude_id with gauge_name
names_path=Path("/inputs/data_updated_2/attributes/attributes_other.csv")
names_file = pd.read_csv(names_path)

# Directories for processed data and plots
output_dir = Path("./processed_data_2/highqual_with_short_estimated")
plots_dir = output_dir / "streamflow_series"
validation_dir = output_dir / "data_validation"
curated_flows_dir = output_dir / "curated_flows"

# Create output directories
output_dir.mkdir(exist_ok=True, parents=True)
plots_dir.mkdir(exist_ok=True, parents=True)
validation_dir.mkdir(exist_ok=True, parents=True)
curated_flows_dir.mkdir(exist_ok=True, parents=True)


In [4]:
def find_long_estimated_periods(quality_series, max_length=10):
    """
    Process quality codes to handle estimated data (code "Dato estimado o calculado"):
    - Keep periods of ≤10 consecutive days
    - Mark longer periods as missing (code 250)
    Returns a Series of processed quality codes.
    """
    # Convert to numpy array for faster processing
    quality_codes = quality_series.values.copy()
    processed_codes = quality_codes.copy()
    
    # Initialize counter for consecutive estimated days
    count = 0
    
    # Process each quality code
    for i in range(len(quality_codes)):
        if quality_codes[i] == "Dato estimado o calculado":
            count += 1
            # When we hit 11 consecutive days
            if count == max_length + 1:
                # Set the previous 10 days to missing
                processed_codes[i-max_length:i] = 250
                # Set current day to missing
                processed_codes[i] = 250
            # For any subsequent days in long periods
            elif count > max_length + 1:
                processed_codes[i] = 250
        else:
            # Reset counter when we see any other quality code
            count = 0
    
    return pd.Series(processed_codes, index=quality_series.index)

In [5]:
def create_quality_validation_plot(df, estimated_mask, gauge_id):
    """Create validation plot showing where short estimated periods were kept."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), height_ratios=[3, 1])
    
    # Plot streamflow
    ax1.plot(df.index, df['Valor'], label='Streamflow (m³/s)', alpha=0.7)
    
    # Highlight kept estimated periods
    estimated_data = df[estimated_mask]
    ax1.scatter(estimated_data.index, estimated_data['Valor'], 
               color='orange', alpha=0.5, label='Kept Estimated Data')

    # row = names_file.loc[names_file["gauge_name"] == gauge_id].iloc[0]
    # gauge_name = row["gauge_id"]

    # --- Fix: gauge_id is actually gauge_name here ---
    match = names_file.loc[names_file["gauge_name"] == gauge_id]
    if not match.empty:
        csv_gauge_id = match.iloc[0]["gauge_id"]
    else:
        csv_gauge_id = "Unknown ID"

    ax1.set_title(f'Gauge: {csv_gauge_id} ({gauge_id}) - Streamflow with Kept Estimated Periods')
    ax1.set_ylabel('Streamflow (m³/s)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot quality codes
    quality_colors = {
        "Dato aceptado": '#2ecc71',   # Good - Green
        # "Serie incompleta": '#f1c40f',   # Incomplete - Yellow
        "Dato estimado o calculado": '#e67e22',  # Estimated - Orange
        250: '#ffffff'   # Missing - White
    }

    # Plot quality bar
    ordered_codes = ["Dato aceptado", "Dato estimado o calculado", 250] #"Augmented data", 250]
    # for code in sorted(quality_colors.keys()):
    for code in ordered_codes:
        mask = df['Índice de calidad'] == code

        if code == "Augmented data":
            mask = (df['Índice de calidad'] == "Dato aceptado") & (df["Índice de revisión"] == "Augmented data")
            
        if mask.any():
            ax2.fill_between(df.index, 0, 1, where=mask, 
                           color=quality_colors[code], alpha=0.7)
    
    # Highlight kept estimated periods
    ax2.fill_between(df.index, 0, 1, where=estimated_mask, 
                    color='#6baed6', alpha=0.3, label='Kept Estimated')
    
    ax2.set_ylim(0, 1)
    ax2.set_xlabel('Date')
    ax2.set_yticks([])
    
    # Add quality code legend
    quality_patches = [plt.Rectangle((0,0),1,1, facecolor=color, edgecolor="black", linewidth=0.6) 
                     for color in quality_colors.values()]
    quality_labels = ['Good', 'Estimated', 'Missing'] #'Augmented', 
    ax2.legend(quality_patches + [plt.Rectangle((0,0),1,1, facecolor='#6baed6', edgecolor="black", linewidth=0.6, alpha=0.3)],
              quality_labels + ['Kept Estimated'],
              ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.2))
    
    plt.tight_layout()
    plt.savefig(validation_dir / f'quality_validation_{gauge_id}.png',
                bbox_inches='tight', dpi=300)
    plt.close()

In [6]:
def process_quality_codes(df, gauge_id):
    """
    Process quality codes with modified criteria:
    - Keep code "Dato aceptado" (good quality)
    - Keep code "Dato estimado o calculado" (estimated) for periods ≤10 days
    - Set all other data ("Serie incompleta","Aproximado por falta de escala", "Afectado por remanso","Dato dudoso") to missing (code 250)
    """
    # Start with the original quality codes
    processed_qc = df['Índice de calidad'].copy()
    
    # Process estimated data (code 100)
    processed_qc = find_long_estimated_periods(processed_qc)
    
    # Set suspect ("Aproximado por falta de escala", "Afectado por remanso","Dato dudoso") to missing (code 250)
    processed_qc[processed_qc == "Serie incompleta"] = 250
    processed_qc[processed_qc == "Aproximado por falta de escala"] = 250
    processed_qc[processed_qc == "Afectado por remanso"] = 250
    processed_qc[processed_qc == "Dato dudoso"] = 250
    processed_qc[processed_qc == 250] = 250
    
    # Create validation plot for this station if it has any estimated data
    estimated_mask = (df['Índice de calidad'] == "Dato estimado o calculado") & (processed_qc != 250)

    if estimated_mask.any():
        create_quality_validation_plot(df, estimated_mask, gauge_id)
    
    # Return mask of data to keep
    return (processed_qc == "Dato aceptado") | (processed_qc == "Dato estimado o calculado")


In [7]:
# Read all streamflow files
all_data = {}
processed_stations = []  # Keep track of all processed stations

print("\nProcessing streamflow files...")
for file in path_to_xls_files.glob("*.xls"):

    gauge_id = file.stem.split(' - ')[0] #mine
    processed_stations.append(gauge_id)
    
    # Read the CSV file
    df = pd.read_excel(file)
    df['Fecha'] = pd.to_datetime(df['Fecha'], format="%d/%m/%Y %H:%M")
    # Set date as index
    df = df.set_index('Fecha')

    # Fill missing dates with NaN to ensure continuous daily series
    full_range = pd.date_range(df.index.min(), df.index.max(), freq='D')
    df = df.reindex(full_range)
    # Fill 'Indice de calidad' with 250 for newly created rows
    df['Índice de calidad'] = df['Índice de calidad'].fillna(250)
    
    # Then process quality codes with modified criteria
    high_quality_mask = process_quality_codes(df, gauge_id)
    
    # Set non-high-quality data to NaN
    df_processed = df.copy()
    df_processed.loc[~high_quality_mask, 'Valor'] = np.nan
    
    # # Clean specific periods for this gauge
    # df_processed = clean_specific_periods(df_processed, gauge_id)

    # --- Add curated streamflow values to original XLS ---
    # Ensure full daily coverage for both datasets
    full_range = pd.date_range(df.index.min(), df.index.max(), freq='D')
    
    # Align original and processed data
    df_full = df.reindex(full_range)
    df_processed_full = df_processed.reindex(full_range)
    
    # Add curated streamflow column
    df_full['Valor_curado'] = df_processed_full['Valor']
    
    # Save to curated flows folder
    output_xls_path = curated_flows_dir / f"{gauge_id}_curated.xls"
    df_full.to_excel(output_xls_path, index_label='Fecha')
    
    print(f"Saved curated Excel for {gauge_id} → {output_xls_path.name}")
    
    all_data[gauge_id] = {
        'Valor': df_processed['Valor'],
        'Índice de calidad': df['Índice de calidad']
    }



Processing streamflow files...
Saved curated Excel for Paso de las Toscas → Paso de las Toscas_curated.xls
Saved curated Excel for Paso Aguiar → Paso Aguiar_curated.xls
Saved curated Excel for Paso de las Piedras → Paso de las Piedras_curated.xls
Saved curated Excel for Sarandi del Yi → Sarandi del Yi_curated.xls
Saved curated Excel for Paso de las Piedras (R3) → Paso de las Piedras (R3)_curated.xls
Saved curated Excel for Paso Mazangano → Paso Mazangano_curated.xls
Saved curated Excel for Fraile Muerto → Fraile Muerto_curated.xls
Saved curated Excel for Paso del Borracho → Paso del Borracho_curated.xls
Saved curated Excel for Durazno → Durazno_curated.xls
Saved curated Excel for Paso Baltasar → Paso Baltasar_curated.xls
Saved curated Excel for Paso de Coelho → Paso de Coelho_curated.xls
Saved curated Excel for Tacuarembo → Tacuarembo_curated.xls
Saved curated Excel for Paso Manuel Diaz → Paso Manuel Diaz_curated.xls
Saved curated Excel for Paso de los Mellizos → Paso de los Mellizos_

In [8]:
# Combine all data into DataFrames
combined_df = pd.DataFrame({gauge_id: data['Valor'] for gauge_id, data in all_data.items()})


In [9]:
filtered_df = combined_df[combined_df.index >= '1980-01-01'].copy()

In [10]:
# Define training, validation, and test periods
train_period = pd.date_range(start='1999-10-01', end='2008-09-30', freq='D') #pd.date_range(start='1989-09-01', end='2001-08-31', freq='D')
val_period = pd.date_range(start='1989-10-01', end='1999-09-30', freq='D') #pd.date_range(start='2001-09-01', end='2005-08-31', freq='D')
test_period = pd.date_range(start='2008-10-01', end='2019-12-31', freq='D') #pd.date_range(start='2005-09-01', end='2009-08-31', freq='D')

In [11]:
# Split data into periods
train_data = filtered_df[filtered_df.index.isin(train_period)]
val_data = filtered_df[filtered_df.index.isin(val_period)]
test_data = filtered_df[filtered_df.index.isin(test_period)]

In [12]:
# Calculate statistics for each station
station_stats = {}
usable_stations = []

print("\nCalculating statistics for natural catchments only...")
for station in filtered_df.columns:  # Now filtered_df only contains natural catchments
    data = filtered_df[station]
    
    # Calculate overall missing ratio
    missing_ratio = data.isnull().mean()
    
    # Calculate yearly missing ratios
    yearly_missing = data.groupby(data.index.year).apply(lambda x: x.isnull().mean())
    bad_years = (yearly_missing > 0.50).sum()  # Count years with >50% missing data

    #calculate period statistics (mine)
    train_data_station = data.loc[data.index.intersection(train_period)]
    val_data_station   = data.loc[data.index.intersection(val_period)]
    test_data_station  = data.loc[data.index.intersection(test_period)]

    train_missing = train_data_station.isnull().mean()
    val_missing = val_data_station.isnull().mean()
    test_missing = test_data_station.isnull().mean()
    
    # Check if there's any data in each period
    has_train_data = not train_data_station.isnull().all()
    has_val_data = not val_data_station.isnull().all()
    has_test_data = not test_data_station.isnull().all()
    
    stats = {
        'total_days': len(data),
        'missing_days': data.isnull().sum(),
        'missing_percentage': missing_ratio * 100,
        'mean_flow': data.mean(),
        'min_flow': data.min(),
        'max_flow': data.max(),
        'bad_years': bad_years,
        'train_missing_ratio': train_missing,
        'val_missing_ratio': val_missing,
        'test_missing_ratio': test_missing,
        'has_train_data': has_train_data,
        'has_val_data': has_val_data,
        'has_test_data': has_test_data
    }
    station_stats[station] = stats
    
    # Calculate available days in each period
    train_available_days = train_data_station.count()  # Count non-NaN values
    val_available_days = val_data_station.count()
    
    # Required days (6 years in training, 2 years in validation)
    REQUIRED_TRAIN_DAYS = 365.25 * 6  # ~2191 days
    REQUIRED_VAL_DAYS = 365.25 * 2    # ~731 days

    # Apply filtering criteria:
    # 1. Overall missing ratio ≤ 50%
    # 2. Must have minimum required days in training and validation periods
    # 3. Test period data is optional
    if (missing_ratio <= 0.50 and 
        train_available_days >= REQUIRED_TRAIN_DAYS and 
        val_available_days >= REQUIRED_VAL_DAYS):
        usable_stations.append(station)



Calculating statistics for natural catchments only...


In [13]:
print(f"\nFound {len(usable_stations)} usable stations")
print("Usable stations:", sorted(usable_stations))

# Print data quality summary
print("\nData Quality Summary:")
print(f"Total catchments processed: {len(filtered_df.columns)}")
print(f"Usable catchments after filtering: {len(usable_stations)}")
print("\nPeriod Information:")
print(f"Training period: {train_period[0].strftime('%Y-%m-%d')} to {train_period[-1].strftime('%Y-%m-%d')}")
print(f"Validation period: {val_period[0].strftime('%Y-%m-%d')} to {val_period[-1].strftime('%Y-%m-%d')}")
print(f"Testing period: {test_period[0].strftime('%Y-%m-%d')} to {test_period[-1].strftime('%Y-%m-%d')}")

# Calculate and print average missing data percentages
print("\nAverage Missing Data Percentages for Usable Stations:")
print(f"Overall: {filtered_df[usable_stations].isnull().mean().mean()*100:.1f}%")
print(f"Training period: {train_data[usable_stations].isnull().mean().mean()*100:.1f}%")
print(f"Validation period: {val_data[usable_stations].isnull().mean().mean()*100:.1f}%")
print(f"Testing period: {test_data[usable_stations].isnull().mean().mean()*100:.1f}%")



Found 11 usable stations
Usable stations: ['Bequelo', 'Fraile Muerto', 'Paso Aguiar', 'Paso Baltasar', 'Paso Manuel Diaz', 'Paso Mazangano', 'Paso de Coelho', 'Paso de las Piedras', 'Paso de las Piedras (R3)', 'Paso de las Toscas', 'Paso del Borracho']

Data Quality Summary:
Total catchments processed: 16
Usable catchments after filtering: 11

Period Information:
Training period: 1999-10-01 to 2008-09-30
Validation period: 1989-10-01 to 1999-09-30
Testing period: 2008-10-01 to 2019-12-31

Average Missing Data Percentages for Usable Stations:
Overall: 19.9%
Training period: 6.8%
Validation period: 6.6%
Testing period: 13.0%


In [14]:
# Save the datasets (mine)
train_data = filtered_df.reindex(train_period, columns=usable_stations)
val_data   = filtered_df.reindex(val_period, columns=usable_stations)
test_data  = filtered_df.reindex(test_period, columns=usable_stations)

train_data.to_csv(output_dir / "train_data.csv")
val_data.to_csv(output_dir / "validation_data.csv")
test_data.to_csv(output_dir / "test_data.csv")
combined_df.to_csv(output_dir / "all_data.csv")

# Save station statistics
stats_df = pd.DataFrame.from_dict(station_stats, orient='index')
stats_df.to_csv(output_dir / "station_statistics.csv")

print("\nProcessing complete!")
print(f"Results saved in: {output_dir}")
print("Check validation_dir for comparison plots and statistics")


Processing complete!
Results saved in: processed_data_2/highqual_with_short_estimated
Check validation_dir for comparison plots and statistics
